Check before running:
1. Check api key is correctly named and stored
2. Check if the examples DMN XML file uploaded to computer with correct file name!
2. check the model name
3. Check the TEMPERATURE
4. Check max tokens
5. Paste the description
6. Change the ID of description

In [ ]:
# ── 0. Setup ───────────────────────────────────────────────
!pip install -q google-genai pandas

from google import genai
from google.genai import types
from google.colab import files
from google.colab import userdata
import pandas as pd
import os
import re

os.makedirs("examples", exist_ok=True)
os.makedirs("experiments/dmn", exist_ok=True)

In [ ]:
# ── 1. Upload example DMN files ────────────────────────
# Upload file:
# example_1.dmn


uploaded = files.upload()

for file_name in uploaded.keys():
    os.rename(file_name, f"examples/{file_name}")

print("Uploaded files:", os.listdir("examples"))

# ── 2. Load and validate example files ─────────────────────
def load_example(file_path: str) -> str:
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Missing file: {file_path}\n"
            "Make sure you uploaded all required files."
        )

    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()

example_1_xml = load_example("examples/example_1.dmn")

Saving example_1.dmn to example_1.dmn
Uploaded files: ['example_1.dmn']


In [ ]:

# ── 3. Insert textual descriptions manually ────────────────
example_1_text = """An institution decides to distribute scholarships but can obviously not give them to everyone. Therefore, they decide to distribute them based on the grades, annual income and whether that person already received any scholarships for the year they are applying to. In short a person can only be eligible for a scholarship if the grades are excellent or good, they earn less than 50000 a year and have not received any other scholarship yet. All the other cases make that the person is not eligible for that scholarship."""

# ── 4. API key and model ───────────────────────────────────
#Upload API key from secrets
#Note: the api key has to be named: GEMINI_API_KEY
API_KEY = userdata.get("GEMINI_API_KEY_2")

# Use the uploaded key
client = genai.Client(api_key=API_KEY)

#This model is taken from AiStudio
MODEL_NAME = "gemini-3-flash-preview"

# ── 5. Configuration ───────────────────────────────────────
# Change this manually for each temperature run: 0.2 / 0.4 / 0.6
TEMPERATURE = 0.2

N_ITERATIONS = 3
MAX_TOKENS = 20000
TOP_P = 1.0

# ── 6. New description for generation ──────────────────────
# Change both description_id and description for each new run.
description_id = "description_id"

description = """enter your description here"""


In [ ]:
# ── 7. Prompt builder ──────────────────────────────────────
def build_few_shot_prompt(description: str) -> str:
    return f"""You are an expert in generating DMN 1.3 XML files compatible with Camunda Modeler.(Persona)

Use the following Camunda-exported DMN example as the structural reference.(Instruction)

Example description: (Context)
{example_1_text}

Example DMN XML:
{example_1_xml}

Generate a complete Camunda-compatible DMN XML file for the new description below.

Requirements:(Constraint instruction)
- Return ONLY raw XML.
- Do not include explanations, comments, headings, labels, markdown, or code fences.
- Start with:
  <?xml version="1.0" encoding="UTF-8"?>
- End with:
  </definitions>

- Include valid DMN 1.3 namespace declarations.
- Include inputData, decision, decisionTable, informationRequirement, and DMNDI elements.
- Every decision must contain exactly one decisionTable.
- Every decisionTable must use hitPolicy="UNIQUE".
- Rules must be mutually exclusive so that no input combination matches more than one rule.

- Use <inputEntry><text>-</text></inputEntry> to represent any value ("don't care").
- Escape XML special characters correctly: &amp; &lt; &gt;.
- Do not use raw "<" inside XML text nodes.
- Use &lt; for less-than comparisons inside <text> elements.
- Do not use undeclared XML entities such as &ge; or &le;.

- Ensure all XML tags are correctly opened and closed.
- Never leave incomplete XML elements.
- Every dmndi:DMNShape must close with </dmndi:DMNShape>.
- Every dmndi:DMNEdge must close with </dmndi:DMNEdge>.

- Follow the XML structure, namespace style, element order, decision table style, and DMNDI style of the example as closely as possible.
- Keep the DMNDI layout simple and valid.
- The file must be importable and readable in Camunda Modeler.

Text: '{description}' [/INST]</s>"""

# ── 8. Clean model output ──────────────────────────────────
def clean_model_output(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```xml\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    xml_start = text.find("<?xml")
    if xml_start == -1:
        xml_start = text.find("<definitions")

    if xml_start != -1:
        text = text[xml_start:]

    xml_end = text.rfind("</definitions>")
    if xml_end != -1:
        text = text[:xml_end + len("</definitions>")]

    return text.strip()


# ── 9. Run generation ──────────────────────────────────────
prompt = build_few_shot_prompt(description)

for iteration in range(1, N_ITERATIONS + 1):
    print(f"▶ Gemini | {description_id} | temp={TEMPERATURE} | iter={iteration}")

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=TEMPERATURE,
            max_output_tokens=MAX_TOKENS,
            top_p=TOP_P,
        ),
    )

    usage = response.usage_metadata

    print("Input tokens:", usage.prompt_token_count)
    print("Output tokens:", usage.candidates_token_count)
    print("Total tokens:", usage.total_token_count)

    dmn_xml = clean_model_output(response.text)

    base_name = f"{description_id}_gemini_few_shot_temp_{TEMPERATURE}_iter_{iteration}"
    dmn_path = f"experiments/dmn/{base_name}.dmn"

    # Save DMN file
    with open(dmn_path, "w", encoding="utf-8") as f:
        f.write(dmn_xml)

    print(f"Saved: {dmn_path}")

# ── 10. Download DMN results ───────────────────────────────
for iteration in range(1, N_ITERATIONS + 1):
    base_name = f"{description_id}_gemini_few_shot_temp_{TEMPERATURE}_iter_{iteration}"
    files.download(f"experiments/dmn/{base_name}.dmn")

▶ Gemini | description_1 | temp=0.2 | iter=1
Input tokens: 3335
Output tokens: 1767
Total tokens: 9666
Saved: experiments/dmn/description_1_gemini_few_shot_temp_0.2_iter_1.dmn


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>